## Environment

In [ ]:
!pip install -q timm transformers accelerate sentencepiece pandas pillow tqdm matplotlib nltk rouge-score bert-score google-genai

## Load shared code

In [ ]:
%run ./02_shared.ipynb

## Gemini API key

In [ ]:
import os

# Set this before running Gemini judge:
# os.environ["GEMINI_API_KEY"] = "YOUR_API_KEY"

if "GEMINI_API_KEY" not in os.environ:
    print("Missing GEMINI_API_KEY. Set it before running Gemini judge.")

## Evaluation configuration

In [ ]:
from dataclasses import dataclass

import torch


@dataclass
class EvalConfig:
    meta_file: str = "../processed/meta.json"
    judge_model: str = "models/gemini-3.1-flash-lite"
    use_gemini_judge: bool = True
    judge_max_samples: int = 100
    judge_batch_size: int = 4
    bertscore_model: str = "vinai/phobert-base-v2"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

## A-model loading

In [ ]:
from pathlib import Path

import torch


class EvalModelLoad:
    def __init__(self, config: AConfig, data: DataModule) -> None:
        self.config = config
        self.data = data
        self.meta = data.meta
        self.builder = ModelBuild(config, data.tokenizer)

    def checkpoint_path(self, decoder: str) -> Path:
        return Path(self.meta["checkpoint_dir"]) / f"a_{decoder}" / "best.pt"

    def load(self, decoder: str) -> VQAModel:
        model = self.builder.build(decoder)
        state = Checkpoint.load(self.checkpoint_path(decoder), self.config.device)

        if state is None:
            raise FileNotFoundError(f"Missing checkpoint: {decoder}")

        model.load_state_dict(state["model"])
        model.to(self.config.device)
        model.eval()

        return model

    def move(self, batch: dict[str, object]) -> dict[str, object]:
        moved = {}

        for key, value in batch.items():
            if isinstance(value, torch.Tensor):
                moved[key] = value.to(self.config.device)
            else:
                moved[key] = value

        return moved

## Prediction

In [ ]:
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm


class PredictRun:
    def __init__(
        self,
        model: VQAModel,
        data: DataModule,
        loader: EvalModelLoad,
        label: str
    ) -> None:
        self.model = model
        self.data = data
        self.loader = loader
        self.label = label
        self.test_items = JsonRead.list_data(self.data.meta["test_json"])
        self.test_dir = Path(self.data.meta["test_json"]).parent

    def resolve_image_path(self, raw_path: object) -> str:
        path = Path(str(raw_path))

        if path.exists():
            return str(path)

        alt_path = self.test_dir / path

        if alt_path.exists():
            return str(alt_path)

        return str(path)

    @torch.no_grad()
    def run(self) -> pd.DataFrame:
        _, _, test_loader = self.data.loaders()
        rows = []
        offset = 0
        progress = tqdm(
            test_loader,
            desc=f"eval {self.label}",
            leave=False
        )

        for batch in progress:
            batch = self.loader.move(batch)
            generated = self.model.generate(
                images=batch["images"],
                question_ids=batch["question_ids"],
                question_mask=batch["question_mask"]
            )
            preds = self.data.tokenizer.batch_decode(
                generated.detach().cpu().tolist(),
                skip_special_tokens=True
            )

            for i, (gold, pred) in enumerate(zip(batch["answers"], preds)):
                item = self.test_items[offset + i]
                rows.append(
                    {
                        "config": self.label,
                        "question": str(item["question"]),
                        "image_path": self.resolve_image_path(
                            item["image_path"]
                        ),
                        "answer": gold,
                        "prediction": pred.strip()
                    }
                )

            offset += len(preds)

        return pd.DataFrame(rows)

## Automatic metrics

In [ ]:
import numpy as np
from bert_score import BERTScorer
from nltk.translate.bleu_score import SmoothingFunction
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer


class AutoMetric:
    def __init__(self, bertscore_model: str) -> None:
        self.exact = ExactMatchScore()
        self.rouge = rouge_scorer.RougeScorer(
            ["rougeL"],
            use_stemmer=False
        )
        self.bleu_smooth = SmoothingFunction().method1
        self.bertscore_model = bertscore_model
        self.bertscorer = None

    def bleu(self, pred: str, gold: str) -> float:
        pred_tokens = str(pred).split()
        gold_tokens = str(gold).split()

        if not pred_tokens or not gold_tokens:
            return 0.0

        return float(
            sentence_bleu(
                [gold_tokens],
                pred_tokens,
                smoothing_function=self.bleu_smooth
            )
        )

    def rouge_l(self, pred: str, gold: str) -> float:
        return float(self.rouge.score(str(gold), str(pred))["rougeL"].fmeasure)

    def meteor(self, pred: str, gold: str) -> float:
        pred_tokens = str(pred).split()
        gold_tokens = str(gold).split()

        if not pred_tokens or not gold_tokens:
            return 0.0

        try:
            return float(meteor_score([gold_tokens], pred_tokens))
        except Exception:
            return 0.0

    def bertscore(self, preds: list[str], golds: list[str]) -> float:
        if not preds:
            return 0.0

        if self.bertscorer is None:
            self.bertscorer = BERTScorer(
                model_type=self.bertscore_model,
                num_layers=12,
                lang="vi",
                rescale_with_baseline=False
            )

        _, _, f1 = self.bertscorer.score(preds, golds)

        return float(f1.mean().item())

    def table(self, pred_table: pd.DataFrame) -> dict[str, float]:
        preds = pred_table["prediction"].astype(str).tolist()
        golds = pred_table["answer"].astype(str).tolist()
        exact = self.exact.batch(preds, golds)["exact_match"]
        bleu = np.mean([self.bleu(p, g) for p, g in zip(preds, golds)])
        rouge_l = np.mean([self.rouge_l(p, g) for p, g in zip(preds, golds)])
        meteor = np.mean([self.meteor(p, g) for p, g in zip(preds, golds)])
        bertscore = self.bertscore(preds, golds)

        return {
            "exact_match": float(exact),
            "bleu": float(bleu),
            "rouge_l": float(rouge_l),
            "meteor": float(meteor),
            "bertscore_f1": float(bertscore),
            "samples": int(len(pred_table))
        }

## Gemini-as-a-judge, 4 samples per request

In [ ]:
import json
import re
import time
from pathlib import Path

import numpy as np
from google import genai
from PIL import Image
from tqdm.auto import tqdm


class GeminiJudge:
    def __init__(self, model_name: str, batch_size: int = 4) -> None:
        if "GEMINI_API_KEY" not in os.environ:
            raise EnvironmentError("Set GEMINI_API_KEY before running Gemini judge.")

        self.model_name = model_name
        self.batch_size = batch_size
        self.client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

    def rubric(self) -> str:
        return (
            "Bạn là giám khảo VQA tiếng Việt. "
            "Hãy nhìn từng ảnh, đọc câu hỏi, đáp án đúng và câu trả lời dự đoán. "
            "Chấm điểm từng mẫu trên thang 0 đến 1 theo rubric sau:\n"
            "1.0 = hoàn toàn đúng với ảnh và tương đương đáp án đúng.\n"
            "0.75 = gần đúng, chỉ thiếu hoặc khác chi tiết nhỏ.\n"
            "0.50 = đúng một phần nhưng thiếu thông tin quan trọng.\n"
            "0.25 = có liên quan nhưng phần lớn sai.\n"
            "0.0 = sai hoặc không liên quan.\n\n"
            "Chỉ trả về JSON array gồm đúng số điểm tương ứng theo thứ tự mẫu. "
            "Ví dụ: [1.0, 0.5, 0.75, 0.0]. Không giải thích."
        )

    def sample_text(self, index: int, row: pd.Series) -> str:
        return (
            f"Mẫu {index}:\n"
            f"Câu hỏi: {row['question']}\n"
            f"Đáp án đúng: {row['answer']}\n"
            f"Dự đoán: {row['prediction']}\n"
        )

    def build_contents(self, rows: pd.DataFrame) -> list[object]:
        contents = [self.rubric()]

        for idx, (_, row) in enumerate(rows.iterrows(), start=1):
            contents.append(self.sample_text(idx, row))
            image = Image.open(Path(str(row["image_path"]))).convert("RGB")
            contents.append(image)

        return contents

    def parse_scores(self, text: str, expected: int) -> list[float]:
        raw = str(text).strip()

        try:
            data = json.loads(raw)
            scores = [float(value) for value in data]
        except Exception:
            values = re.findall(r"(?<!\d)(?:1(?:\.0)?|0(?:\.\d+)?)(?!\d)", raw)
            scores = [float(value) for value in values]

        if len(scores) < expected:
            scores.extend([0.0] * (expected - len(scores)))

        scores = scores[:expected]

        return [
            min(1.0, max(0.0, score))
            for score in scores
        ]

    def score_batch(self, rows: pd.DataFrame) -> list[float]:
        response = self.client.models.generate_content(
            model=self.model_name,
            contents=self.build_contents(rows)
        )

        return self.parse_scores(response.text, len(rows))

    def score_table(self, table: pd.DataFrame, max_samples: int) -> float:
        if table.empty:
            return 0.0

        sample = table.head(max_samples).reset_index(drop=True)
        scores = []

        for start in tqdm(
            range(0, len(sample), self.batch_size),
            desc="gemini judge",
            leave=False
        ):
            batch = sample.iloc[start:start + self.batch_size]

            try:
                batch_scores = self.score_batch(batch)
            except Exception as exc:
                print("Gemini judge error:", exc)
                batch_scores = [0.0] * len(batch)

            scores.extend(batch_scores)
            time.sleep(0.2)

        return float(np.mean(scores)) if scores else 0.0

## Full metric plot

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt


class EvalPlot:
    def __init__(self, figure_dir: str) -> None:
        self.figure_dir = Path(figure_dir)
        self.metric_map = {
            "exact_match": "Exact Match",
            "bleu": "BLEU",
            "rouge_l": "ROUGE-L",
            "meteor": "METEOR",
            "bertscore_f1": "BERTScore F1",
            "gemini_judge": "Gemini Judge"
        }

    def metric_compare(self, table: pd.DataFrame, out_name: str) -> Path | None:
        available = [
            col
            for col in self.metric_map
            if col in table.columns
        ]

        if table.empty or not available:
            return None

        out_path = self.figure_dir / out_name
        plot_table = table.set_index("config")[available].T
        plot_table.index = [
            self.metric_map[col]
            for col in plot_table.index
        ]

        fig, ax = plt.subplots(figsize=(12, 6))
        plot_table.plot(kind="bar", ax=ax)
        ax.set_ylim(0, 1.0)
        ax.set_xlabel("Metric")
        ax.set_ylabel("Score")
        ax.set_title("A1 vs A2 - Full Metric Comparison")
        ax.grid(True, axis="y", alpha=0.3)
        ax.legend(title="Model")
        ax.set_xticks(range(len(plot_table.index)))
        ax.set_xticklabels(
            plot_table.index,
            rotation=0,
            ha="center"
        )
        fig.tight_layout()
        out_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_path, dpi=160, bbox_inches="tight")
        plt.close(fig)

        return out_path

## A1/A2 evaluation runner

In [ ]:
from pathlib import Path

import pandas as pd


class EvalRunner:
    def __init__(self, config: EvalConfig) -> None:
        self.config = config
        self.a_config = AConfig(meta_file=config.meta_file)
        self.data = DataModule(self.a_config)
        self.meta = self.data.meta
        self.loader = EvalModelLoad(self.a_config, self.data)
        self.metric = AutoMetric(config.bertscore_model)
        self.plot = EvalPlot(self.meta["figure_dir"])
        self.judge = None

    def add_judge(self, pred_table: pd.DataFrame, result: dict[str, object]) -> None:
        if not self.config.use_gemini_judge:
            return

        if self.judge is None:
            self.judge = GeminiJudge(
                model_name=self.config.judge_model,
                batch_size=self.config.judge_batch_size
            )

        result["gemini_judge"] = self.judge.score_table(
            pred_table,
            self.config.judge_max_samples
        )

    def evaluate_model(
        self,
        label: str,
        model: VQAModel,
        decoder: str
    ) -> pd.DataFrame:
        pred_table = PredictRun(
            model=model,
            data=self.data,
            loader=self.loader,
            label=label
        ).run()
        result = self.metric.table(pred_table)
        result["config"] = label
        result["decoder"] = decoder
        self.add_judge(pred_table, result)

        return pd.DataFrame([result])

    def save_result(self, table: pd.DataFrame, name: str) -> None:
        path = Path(self.meta["result_dir"]) / name
        path.parent.mkdir(parents=True, exist_ok=True)
        table.to_csv(path, index=False)

    def run(self) -> pd.DataFrame:
        a1 = self.evaluate_model(
            "A1",
            self.loader.load("lstm"),
            "lstm"
        )
        a2 = self.evaluate_model(
            "A2",
            self.loader.load("transformer"),
            "transformer"
        )
        compare = pd.concat([a1, a2], ignore_index=True)
        self.save_result(compare, "a_compare.csv")
        self.plot.metric_compare(compare, "a_metric_compare.png")
        MemoryCleaner.clear()

        return compare

## Run A1/A2 evaluation

In [ ]:
config = EvalConfig()
runner = EvalRunner(config)
a_compare = runner.run()
a_compare